# Model selection: LightGBM, CatBoost и XGBoost

**Задача.** Прогнозировать `boardings` для каждой комбинации `(route, date, hour)` по известным заранее календарным признакам и замороженной погодной климатологии. Сравниваем текущий frozen LightGBM как контроль, CatBoostRegressor и XGBRegressor как две другие семьи градиентного бустинга. На этом этапе параметры фиксированы; HPO не выполняется. Итогом будут не более двух комбинаций для следующего этапа HPO.

Код обучения и метрик находится в `ml/src/model_selection.py`. Этот notebook вызывает его функции на исходных model-ready CSV, заново обучает все шесть комбинаций и сохраняет тот же JSON, что CLI `python ml/src/model_selection.py`.

## 1. Загрузка и проверка model-ready данных

Берём только исторический `model_ready_train_2025_jan_oct.csv` с weather columns. Проверяем ключ, полную почасовую сетку, route 5, отсутствие пропусков и точное соответствие отдельным Jan–Aug и Sep–Oct CSV. Nov–Dec здесь не читается: фактических `boardings` за этот период нет, поэтому использовать его для model selection нельзя. Наблюдаемая погода 2025 не входит в признаки из-за утечки будущей информации.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'ml/src/model_selection.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(root / 'ml/src'))
import model_selection as ms

rows = ms.load_data()
display(pd.DataFrame([{'rows': len(rows), 'first_date': rows.date.min().date(),
                       'last_date': rows.date.max().date(),
                       'routes': rows.route.nunique(),
                       'weather_bucket_coverage': rows.temperature_bucket.notna().mean()}]))

,rows,first_date,last_date,routes,weather_bucket_coverage
0,72960,2025-01-01,2025-10-31,10,1.0


## 2. Две одинаковые по смыслу конфигурации признаков

`SET_FEATURES` — принятый core из pilot v3. Проверяем два ранее определённых candidate набора: `SET_FEATURES + WEATHER_FEATURES` и `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES`. `WEATHER_FEATURES` содержит только `temperature_bucket`; пять calendar flags не становятся частью accepted registry. Новых признаков нет.

LightGBM получает фиксированные pandas categories и список categorical columns. CatBoost получает те же категории как строковые метки. XGBoost использует pandas categories и native categorical splits. Преобразование отличается под API модели, но каждая колонка обозначает тот же маршрут, день, час, взаимодействие или температурный интервал; кодирование не является новым feature experiment. Числовые `is_night` и calendar flags сохраняются числовыми.

In [2]:
display(pd.DataFrame([{'configuration': name, 'features': ', '.join(features),
                       'feature_count': len(features)}
                      for name, features in ms.FEATURE_CONFIGURATIONS.items()]))
display(pd.DataFrame([{'family': family, 'preprocessing': description}
                      for family, description in {
                          'LightGBM': 'fixed pandas categories',
                          'CatBoostRegressor': 'category labels as strings',
                          'XGBRegressor': 'native categorical hist splits',
                      }.items()]))

,configuration,features,feature_count
0,weather_only,"route, weekday, hour, route_hour, hour_weekend...",7
1,calendar_weather,"route, weekday, hour, route_hour, hour_weekend...",12


,family,preprocessing
0,LightGBM,fixed pandas categories
1,CatBoostRegressor,category labels as strings
2,XGBRegressor,native categorical hist splits


## 3. Временная валидация

Random split смешал бы будущие и прошлые даты и переоценил качество прогноза. Используем три одинаковых для всех моделей расширяющихся train периода и непересекающиеся validation периоды:

1. Jan–Apr → May–Jun: прогноз начала лета из четырёх месяцев истории.
2. Jan–Jun → Jul–Aug: прогноз второй части лета из шести месяцев.
3. Jan–Aug → Sep–Oct: прогноз осени из восьми месяцев.

В каждом fold признаки известны к его forecast origin. Target validation нужен только после `predict` для расчёта ошибки.

In [3]:
folds = [(spec, *ms.make_fold(rows, spec)) for spec in ms.FOLDS]
display(pd.DataFrame([{'fold': spec[0], 'train': f'{spec[1]} … {spec[2]}',
                       'validation': f'{spec[3]} … {spec[4]}',
                       'train_rows': len(train), 'validation_rows': len(valid)}
                      for spec, train, valid in folds]))

,fold,train,validation,train_rows,validation_rows
0,jan_apr_to_may_jun,2025-01-01 … 2025-04-30,2025-05-01 … 2025-06-30,28800,14640
1,jan_jun_to_jul_aug,2025-01-01 … 2025-06-30,2025-07-01 … 2025-08-31,43440,14880
2,jan_aug_to_sep_oct,2025-01-01 … 2025-08-31,2025-09-01 … 2025-10-31,58320,14640


## 4. Воспроизведение frozen LightGBM

Сначала заново обучаем frozen LightGBM на обоих feature sets и сверяем WAPE каждого fold с сохранёнными weather и pilot v3 experiment artifacts. Существенное расхождение остановит notebook до обучения новых моделей. Предсказания ограничиваются снизу нулём, округляются как в pilot v3 и для route 5 принудительно равны `0`: таков действующий inference contract для маршрута без положительной истории в canonical labels.

In [4]:
all_results = {family: {} for family in ms.FAMILIES}
for configuration in ms.FEATURE_CONFIGURATIONS:
    all_results['LightGBM'][configuration] = [
        ms.evaluate_fold(train, valid, configuration, 'LightGBM')
        for _, train, valid in folds
    ]
reference_checks = ms.check_lightgbm_reference(all_results['LightGBM'])
display(pd.DataFrame([{'configuration': config, 'fold': spec[0],
                       'WAPE': result['wape'],
                       'reference_delta': reference_checks[config][spec[0]]}
                      for config, results in all_results['LightGBM'].items()
                      for (spec, _, _), result in zip(folds, results)]).round(6))

,configuration,fold,WAPE,reference_delta
0,weather_only,jan_apr_to_may_jun,0.199760,0.0
1,weather_only,jan_jun_to_jul_aug,0.191758,0.0
2,weather_only,jan_aug_to_sep_oct,0.113909,0.0
3,calendar_weather,jan_apr_to_may_jun,0.156927,0.0
4,calendar_weather,jan_jun_to_jul_aug,0.201950,0.0
5,calendar_weather,jan_aug_to_sep_oct,0.115756,0.0


## 5. Обучение CatBoost и XGBoost

Для каждой новой семьи берём **те же** train и validation строки и обе конфигурации. CatBoost и XGBoost используют фиксированные параметры одного контрольного запуска, без поиска гиперпараметров. После обучения получаем validation predictions, применяем общее округление и правило route 5, затем рассчитываем метрики. Время обучения и inference измеряются отдельно для каждого fold; значения зависят от компьютера.

In [5]:
for family in ms.FAMILIES[1:]:
    for configuration in ms.FEATURE_CONFIGURATIONS:
        all_results[family][configuration] = [
            ms.evaluate_fold(train, valid, configuration, family)
            for _, train, valid in folds
        ]
display(pd.DataFrame([{'family': family, 'configuration': config,
                       'May–Jun WAPE': results[0]['wape'],
                       'Jul–Aug WAPE': results[1]['wape'],
                       'Sep–Oct WAPE': results[2]['wape']}
                      for family, configs in all_results.items()
                      for config, results in configs.items()]).round(6))

,family,configuration,May–Jun WAPE,Jul–Aug WAPE,Sep–Oct WAPE
0,LightGBM,weather_only,0.199760,0.191758,0.113909
1,LightGBM,calendar_weather,0.156927,0.201950,0.115756
2,CatBoostRegressor,weather_only,0.243121,0.230541,0.184104
3,CatBoostRegressor,calendar_weather,0.203010,0.240489,0.179241
4,XGBRegressor,weather_only,0.210687,0.190402,0.115108
5,XGBRegressor,calendar_weather,0.153780,0.198772,0.115607


## 6. Метрики и сравнение

`WAPE = Σ|actual − prediction| / Σactual`; `WAPE-score = max(0, 1 − WAPE)`. Pooled WAPE складывает абсолютные ошибки **всех трёх непересекающихся validation периодов** и делит на сумму их фактических посадок. Это не среднее трёх WAPE: периоды с разным объёмом посадок получают вес по знаменателю. Mean и median WAPE показывают типичное качество fold, а range (`max − min`) — разброс. Для маршрутов считаем pooled WAPE тем же способом; у route 5 знаменатель нулевой, поэтому WAPE не определён.

Кандидатов ранжируем прежде всего по pooled WAPE, затем смотрим разброс между folds и сильное ухудшение отдельных маршрутов. Хороший pooled результат при провале одного периода или важного маршрута требует осторожности. Время служит дополнительной практической характеристикой, а не заменой качества.

In [6]:
result = ms.build_result(all_results, reference_checks)
summary_rows = []
for item in result['ranking_by_pooled_wape']:
    family, config = item['family'], item['configuration']
    summary = result['summaries'][family][config]
    folds_result = all_results[family][config]
    summary_rows.append({
        'family': family, 'configuration': config,
        'May–Jun WAPE': folds_result[0]['wape'],
        'Jul–Aug WAPE': folds_result[1]['wape'],
        'Sep–Oct WAPE': folds_result[2]['wape'],
        'pooled WAPE': summary['pooled_wape'],
        'WAPE-score': summary['wape_score'],
        'pooled absolute error': summary['pooled_absolute_error'],
        'mean WAPE': summary['mean_wape'], 'median WAPE': summary['median_wape'],
        'fold WAPE range': summary['fold_wape_range'],
        'train seconds': summary['train_seconds'],
        'inference seconds': summary['inference_seconds'],
    })
final_table = pd.DataFrame(summary_rows)
display(final_table.round(6))
route_table = pd.DataFrame([
    {'family': family, 'configuration': config, 'route': route,
     'pooled route WAPE': metrics['wape']}
    for family, configs in result['summaries'].items()
    for config, summary in configs.items()
    for route, metrics in summary['per_route'].items()
]).pivot(index=['family', 'configuration'], columns='route', values='pooled route WAPE')
display(route_table.round(4))

,family,configuration,May–Jun WAPE,Jul–Aug WAPE,Sep–Oct WAPE,pooled WAPE,WAPE-score,pooled absolute error,mean WAPE,median WAPE,fold WAPE range,train seconds,inference seconds
0,XGBRegressor,calendar_weather,0.153780,0.198772,0.115607,0.153647,0.846353,5381944,0.156053,0.153780,0.083165,19.310049,0.615233
1,LightGBM,calendar_weather,0.156927,0.201950,0.115756,0.155712,0.844288,5454275,0.158211,0.156927,0.086193,2.828760,0.638139
2,LightGBM,weather_only,0.199760,0.191758,0.113909,0.166051,0.833949,5816456,0.168476,0.191758,0.085851,2.631342,0.626256
3,XGBRegressor,weather_only,0.210687,0.190402,0.115108,0.169678,0.830322,5943487,0.172066,0.190402,0.095579,19.598252,0.669589
4,CatBoostRegressor,calendar_weather,0.203010,0.240489,0.179241,0.205823,0.794177,7209569,0.207580,0.203010,0.061248,89.308079,0.245451
5,CatBoostRegressor,weather_only,0.243121,0.230541,0.184104,0.217782,0.782218,7628494,0.219255,0.230541,0.059018,81.573781,0.195316


route                                    1      11      12      17      25  \
family            configuration                                              
CatBoostRegressor calendar_weather  0.1558  0.1896  0.1624  0.1669  0.2340   
                  weather_only      0.1664  0.1959  0.1744  0.1901  0.2291   
LightGBM          calendar_weather  0.1261  0.1284  0.1204  0.1200  0.1956   
                  weather_only      0.1431  0.1386  0.1339  0.1133  0.2016   
XGBRegressor      calendar_weather  0.1223  0.1258  0.1185  0.1139  0.1963   
                  weather_only      0.1428  0.1410  0.1360  0.1237  0.2018   

route                                   26      28   5      50       7  
family            configuration                                         
CatBoostRegressor calendar_weather  0.2378  0.2082 NaN  0.2877  0.3164  
                  weather_only      0.2392  0.2177 NaN  0.3099  0.3159  
LightGBM          calendar_weather  0.1651  0.1840 NaN  0.2148  0.2696  
                  weather_only      0.1864  0.1974 NaN  0.2440  0.2831  
XGBRegressor      calendar_weather  0.1686  0.1849 NaN  0.2157  0.2698  
                  weather_only      0.1843  0.2024 NaN  0.2436  0.2871

## 7. Результат experiment infrastructure

Сохраняем параметры, SHA-256 исторических CSV и эталонных результатов, версии библиотек, метрики folds и маршрутов, времена и ранжирование в `ml/experiments/results/model_selection.json`. Запуск CLI использует те же функции. Nov–Dec predictions здесь не создаются и не влияют на выбор.

In [7]:
ms.save_result(result)
print('Saved:', ms.ARTIFACT.relative_to(root).as_posix())
print('HPO_CANDIDATES:', [(item['family'], item['configuration'])
                          for item in result['hpo_candidates']])

Saved: ml/experiments/results/model_selection.json
HPO_CANDIDATES: [('XGBRegressor', 'calendar_weather'), ('LightGBM', 'calendar_weather')]


## Вывод model-selection stage

`HPO_CANDIDATES`: **XGBRegressor + calendar_weather** и **LightGBM + calendar_weather**. Их pooled WAPE — около 0.15365 и 0.15571. XGBoost немного лучше LightGBM на всех трёх folds; разброс между folds близок. На pooled per-route WAPE максимальное ухудшение XGBoost относительно LightGBM составляет около 0.0035 на route 26, без сильного route-level degradation. CatBoost при этих фиксированных параметрах заметно уступает по pooled WAPE. Это только отбор кандидатов; accepted feature registry не меняется, HPO и финальный Nov–Dec fit не выполнялись.